# Lab 00 · Arrume a casa antes de modelar
**Machine Learning na Prática · Dia 1 · Módulo 01: Dados & Datasets** · ~35 min

Neste notebook você vai fazer o que todo projeto de ML faz antes de treinar qualquer modelo:

| Etapa | O que fazer | Tempo |
|---|---|---|
| 1. Inspecionar | formato, tipos, estatísticas, faltantes | 8 min |
| 2. Limpar | duplicados, categorias, outliers, alvo faltante | 10 min |
| 3. Explorar | histogramas, boxplot, correlação | 8 min |
| 4. Preparar | split, one-hot, padronização e `Pipeline` sem vazamento | 9 min |

Procure as células marcadas com **`# TODO`** e substitua os `___` pelo código certo.
Rode as células em ordem (`Shift + Enter`).

## 0 · Preparação

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (8, 4)
print('Tudo pronto!')

### O dataset: desempenho de estudantes
Os dados abaixo são **sintéticos** (gerados aqui mesmo, para não depender de download), mas imitam problemas reais:
alguém digitou errado, algumas linhas foram copiadas duas vezes, o turno foi escrito de jeitos diferentes…

Rode a célula para criar o arquivo `alunos.csv`. Você **não precisa** entender o código de geração agora.

In [ ]:
def gerar_dados(n=400, seed=42):
    rng = np.random.default_rng(seed)
    horas = np.clip(rng.normal(6, 2.2, n), 0.5, 14).round(1)
    faltas = rng.poisson(3, n)
    turno = rng.choice(['manhã', 'tarde', 'noite'], n, p=[.45, .3, .25])
    monitoria = rng.choice(['sim', 'não'], n, p=[.35, .65])
    idade = rng.integers(17, 31, n)
    renda = np.round(rng.lognormal(8.2, 0.5, n), -1)          # R$ por mês, escala grande
    nota = (2.0 + 0.45 * horas - 0.25 * faltas + 0.6 * (monitoria == 'sim')
            - 0.4 * (turno == 'noite') + rng.normal(0, 0.9, n))
    nota = np.clip(nota, 0, 10).round(1)
    df = pd.DataFrame({'horas_estudo': horas, 'faltas': faltas, 'turno': turno,
                       'monitoria': monitoria, 'idade': idade, 'renda_familiar': renda,
                       'nota_final': nota})
    df['situacao'] = np.where(df['nota_final'] >= 6, 'aprovado', 'reprovado')

    # ---- problemas do mundo real ----
    sujo = rng.choice(n, 40, replace=False)
    df.loc[sujo[:14], 'turno'] = rng.choice(['Manhã', 'MANHÃ ', 'manha', 'Noite', ' tarde'], 14)
    df.loc[rng.choice(n, 20, replace=False), 'horas_estudo'] = np.nan
    df.loc[rng.choice(n, 15, replace=False), 'faltas'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), 'nota_final'] = np.nan
    outl = rng.choice(n, 4, replace=False)
    df.loc[outl, 'horas_estudo'] = df.loc[outl, 'horas_estudo'].fillna(7) * 10   # vírgula esquecida
    df = pd.concat([df, df.sample(12, random_state=seed)], ignore_index=True)   # duplicados
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

gerar_dados().to_csv('alunos.csv', index=False)
print('alunos.csv criado!')

## 1 · Inspecionar

Carregue o arquivo e olhe as primeiras linhas.

In [ ]:
# TODO: leia o arquivo 'alunos.csv' com o pandas
df = ___
df.head()

In [ ]:
# TODO: quantas linhas e colunas? E quais os tipos de cada coluna?
print(df.___)
df.___()

In [ ]:
# TODO: estatísticas das colunas numéricas. Repare no máximo de horas_estudo!
df.___()

In [ ]:
# TODO: quantos valores faltantes há em cada coluna?
df.___().sum()

In [ ]:
# TODO: quais valores aparecem na coluna 'turno'?
df['turno'].___()

**Para pensar:** liste os problemas que você encontrou. Quais colunas têm faltantes? O que há de estranho em `turno` e em `horas_estudo`?

*Sua resposta:* …

## 2 · Limpar

### 2.1 Duplicados

In [ ]:
# TODO: conte as linhas duplicadas e remova-as
print('duplicados:', df.duplicated().___())
df = df.___()
print('linhas agora:', len(df))

### 2.2 Categorias inconsistentes
Para o computador, `"manhã"`, `"Manhã"` e `"MANHÃ "` são três valores diferentes.
Padronize: tire os espaços das pontas, deixe tudo minúsculo e corrija `"manha"` sem acento.

In [ ]:
# TODO: padronize a coluna turno
df['turno'] = df['turno'].str.___().str.___()
df['turno'] = df['turno'].replace({'manha': 'manhã'})
df['turno'].value_counts()

### 2.3 Outliers
Ninguém estuda 70 horas *extras* por semana: provavelmente alguém esqueceu a vírgula (7,0 → 70).
Como **não temos certeza**, vamos transformar esses valores em faltantes (`NaN`). Depois eles serão preenchidos como os outros faltantes.

In [ ]:
plt.boxplot(df['horas_estudo'].dropna())
plt.title('horas_estudo antes da limpeza'); plt.show()

# TODO: transforme em NaN os valores de horas_estudo acima de 24
df.loc[df['horas_estudo'] > ___, 'horas_estudo'] = np.nan
df['horas_estudo'].describe()

### 2.4 Alvo faltante
Uma linha sem `nota_final` não ensina nada ao modelo: remova-a.
Os faltantes nas **features** (horas, faltas) **não** serão preenchidos agora — isso será feito dentro do `Pipeline`, usando só o treino (veja a etapa 4).

In [ ]:
# TODO: remova as linhas sem nota_final
df = df.dropna(subset=[___])
print(len(df), 'linhas')
df.isna().sum()

## 3 · Explorar

In [ ]:
# TODO: histogramas de todas as colunas numéricas
df.___(bins=20, figsize=(12, 6))
plt.tight_layout(); plt.show()

**Média × mediana:** compare as duas para `horas_estudo` e para `renda_familiar`. Em qual delas a diferença é maior? Por quê?

In [ ]:
for col in ['horas_estudo', 'renda_familiar']:
    # TODO: calcule a média e a mediana da coluna
    print(f"{col}: média = {df[col].___():.1f} | mediana = {df[col].___():.1f}")

In [ ]:
# TODO: gráfico de dispersão horas_estudo × nota_final
plt.scatter(df[___], df[___], alpha=.5)
plt.xlabel('horas de estudo'); plt.ylabel('nota final'); plt.show()

In [ ]:
# TODO: correlação entre as colunas numéricas
df.___(numeric_only=True).round(2)

In [ ]:
# TODO: nota média por turno e por monitoria
df.groupby(___)['nota_final'].mean().round(2)

**Para pensar:** quais features parecem ajudar a prever a nota? Alguma parece não ter relação nenhuma?
Lembre: correlação não é causalidade.

*Sua resposta:* …

## 4 · Preparar sem vazamento
### 4.1 Cuidado com a coluna que “conta a resposta”
A coluna `situacao` (aprovado/reprovado) é **calculada a partir da nota**. Ela não estaria disponível antes de sabermos a nota,
então usá-la como feature seria **vazamento de dados**. Deixe-a de fora.

In [ ]:
alvo = 'nota_final'
# TODO: escolha as features (todas menos o alvo e a coluna que vaza)
X = df.drop(columns=[alvo, ___])
y = df[alvo]

num = ['horas_estudo', 'faltas', 'idade', 'renda_familiar']
cat = ['turno', 'monitoria']
X.head()

### 4.2 Separar treino e teste **antes** de qualquer transformação

In [ ]:
# TODO: 80% treino, 20% teste, random_state=42
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=___, random_state=42)
print(X_tr.shape, X_te.shape)

### 4.3 O `Pipeline`
O `ColumnTransformer` aplica um tratamento por tipo de coluna:
- **numéricas:** preencher faltantes com a mediana → padronizar (`z = (x − μ) / σ`)
- **categóricas:** one-hot

Dentro do `Pipeline`, mediana, média e desvio são calculados **só com o treino**, inclusive em cada rodada da validação cruzada.

In [ ]:
preparo = ColumnTransformer([
    # TODO: complete com SimpleImputer(strategy='median') e StandardScaler()
    ('num', Pipeline([('imputar', ___), ('padronizar', ___)]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
])
modelo = Pipeline([('preparo', preparo), ('regressao', LinearRegression())])
modelo

### 4.4 Validação cruzada no treino, avaliação final no teste

In [ ]:
# TODO: validação cruzada com 5 folds, só no treino
mae_cv = -cross_val_score(modelo, X_tr, y_tr, cv=___, scoring='neg_mean_absolute_error')
print('MAE por fold:', mae_cv.round(2))
print(f'MAE médio: {mae_cv.mean():.2f} ± {mae_cv.std():.2f}')

In [ ]:
# TODO: treine no treino inteiro e avalie UMA vez no teste
modelo.fit(___, ___)
pred = modelo.predict(X_te)
print(f'MAE no teste: {mean_absolute_error(y_te, pred):.2f} ponto')
print(f'R² no teste:  {r2_score(y_te, pred):.2f}')

## 5 · Desafios extras (se sobrar tempo)
### 5.1 Quanto vale um vazamento?
Treine o mesmo pipeline **incluindo** a coluna `situacao` (como categórica) e compare o R² no teste. O que aconteceu? Por que esse resultado é enganoso?

In [ ]:
# TODO: inclua 'situacao' nas features categóricas e compare
X2 = df.drop(columns=[alvo])
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.2, random_state=42)
preparo2 = ColumnTransformer([
    ('num', Pipeline([('imputar', SimpleImputer(strategy='median')), ('padronizar', StandardScaler())]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat + [___]),
])
vazado = Pipeline([('preparo', preparo2), ('regressao', LinearRegression())]).fit(X2_tr, y2_tr)
print(f'R² com vazamento: {r2_score(y2_te, vazado.predict(X2_te)):.2f}')

### 5.2 A escala importa (para o KNN)
`renda_familiar` está em milhares de reais; `faltas`, em unidades. Compare um **KNN** com e sem padronização.
Qual se sai melhor? Por quê? (Dica: lembre do slide “Quem é mais parecido com você?”.)

In [ ]:
def knn(padronizar):
    passos = [('imputar', SimpleImputer(strategy='median'))]
    if padronizar:
        passos.append(('padronizar', StandardScaler()))
    prep = ColumnTransformer([('num', Pipeline(passos), num),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
    return Pipeline([('preparo', prep), ('knn', KNeighborsRegressor(n_neighbors=15))])

for p in [False, True]:
    # TODO: calcule o MAE médio da validação cruzada (cv=5) para cada versão
    mae = -cross_val_score(knn(p), X_tr, y_tr, cv=5, scoring=___).mean()
    print(f'padronizar={p}: MAE = {mae:.2f}')

## Para discutir
1. Qual decisão de limpeza que você tomou muda o resultado do modelo? (Ex.: transformar outliers em `NaN` em vez de dividir por 10.)
2. Por que preencher os faltantes **dentro** do pipeline, e não antes do `train_test_split`?
3. Se o modelo erra em média ~0,7 ponto, isso é bom ou ruim? Depende de quê?